In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv()                      # reads .env, so the key is available
client = anthropic.Anthropic()     # finds ANTHROPIC_API_KEY automatically

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=200,
    messages=[
        {"role": "user", "content": "In one sentence, what is a milkrun in first-mile logistics?"}
    ],
)

print(response.content[0].text)
print(response.usage)

cost_usd = (response.usage.input_tokens * 0.10 + response.usage.output_tokens * 0.50) / 1_000_000
print(f"Cost: ${cost_usd:.6f}")

A milk run in first-mile logistics is a pickup route in which a single truck or vehicle travels a fixed loop, collecting goods from several suppliers in sequence, rather than making a separate dedicated trip to each one.
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=30, output_tokens=73, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard')
Cost: $0.000039


In [2]:
import sqlite3

conn = sqlite3.connect("logistics.db")

def get_schema(conn):
    """List every table with its columns and types."""
    lines = []
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    for (table,) in tables:
        cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
        col_text = ", ".join(f"{c[1]} {c[2]}" for c in cols)
        lines.append(f"{table}({col_text})")
    return "\n".join(lines)

print(get_schema(conn))

exceptions(exception_id TEXT, run_id TEXT, shipment_id TEXT, exception_type TEXT, exception_date TEXT, resolved_date TEXT)
forecasts(shipper_id TEXT, forecast_date TEXT, forecast_packages INTEGER)
holidays(country TEXT, holiday_date TEXT, holiday_name TEXT)
nodes(node_id TEXT, node_type TEXT, country TEXT, city TEXT, daily_capacity_pallets INTEGER)
run_stops(run_id TEXT, shipper_id TEXT, stop_sequence INTEGER, planned_pickup_time TEXT, actual_pickup_time TEXT, pallets_picked INTEGER)
shipments(shipment_id TEXT, shipper_id TEXT, run_id TEXT, node_id TEXT, service_level TEXT, created_date TEXT, promised_date TEXT, delivered_to_node_date TEXT, status TEXT, hold_reason TEXT, packages INTEGER, pallets INTEGER)
shippers(shipper_id TEXT, name TEXT, country TEXT, city TEXT, onboarded_date TEXT, status TEXT, delisted_date TEXT, operating_days TEXT, ships_on_holidays INTEGER, account_manager TEXT)
truck_runs(run_id TEXT, run_date TEXT, run_type TEXT, carrier TEXT, node_id TEXT, planned_pickup_ti

In [ ]:
CATEGORY_COLUMNS = {
    "shipments": ["status", "hold_reason", "service_level"],
    "truck_runs": ["run_type", "run_outcome"],
    "shippers": ["country", "status", "operating_days"],
    "nodes": ["node_type"],
    "exceptions": ["exception_type"],
}

def get_allowed_values(conn):
    """List the exact values each category column can take."""
    lines = []
    for table, cols in CATEGORY_COLUMNS.items():
        for col in cols:
            rows = conn.execute(
                f"SELECT DISTINCT {col} FROM {table} WHERE {col} IS NOT NULL ORDER BY 1"
            ).fetchall()
            values = [r[0] for r in rows]
            lines.append(f"{table}.{col}: {values}")
    return "\n".join(lines)

print(get_allowed_values(conn))

context = get_schema(conn) + "\n\n" + get_allowed_values(conn)
print(f"\nApprox tokens: {len(context) // 4}")



shipments.status: ['Cancelled', 'Created', 'Delivered to Node', 'In Transit', 'On Hold', 'Picked Up']
shipments.hold_reason: ['At Customs', 'Shipper Compliance']
shipments.service_level: ['1DD', 'NDD']
truck_runs.run_type: ['direct', 'milkrun']
truck_runs.run_outcome: ['completed', 'wasted']
shippers.country: ['ES', 'FR', 'IT', 'UK']
shippers.status: ['active', 'de-listed']
shippers.operating_days: ['Mon-Fri', 'Mon-Sat', 'Mon-Sun', 'Tue-Sun']
nodes.node_type: ['delivery_station', 'warehouse']
exceptions.exception_type: ['Ad-hoc truck request', 'Customs hold', 'Damaged freight', 'Late arrival at node', 'Node capacity rejection', 'Partial pickup (shipper not ready)', 'Shipper closed on arrival (wasted truck)', 'Truck no-show', 'Weather site closure']

Approx tokens: 486


In [10]:
print(len(context)//4)

486
